In [3]:
from pathlib import Path
from pyhere import here
import json
import os
import re
import scanpy as sc

import numpy as np
import pandas as pd
from rasterio import Affine

# from loopy.sample import Sample
# from loopy.utils.utils import remove_dupes, Url

import spatialdata as sd
from spatialdata_io import xenium

#spot_diameter_m = 55e-6 # 55-micrometer diameter for Visium spot
img_channels = "rgb"
default_channels = {"blue": "DAPI"}
default_gene = "ACE2"

import spatialdata as sd
from spatialdata_io import xenium


/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:

# Define the path to your Xenium dataset directory
zarr_path = "xenium_2.0.0_io/data.zarr"

# Load the data into a SpatialData object
sdata = sd.read_zarr(zarr_path)

# The 'sdata' object now contains your Xenium data, including images,
# transcripts, and cell segmentation information, organized within the
# SpatialData framework.

version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/Users/kevinstachelek/mamba/envs/samui/lib/python3.10/site-packages/zarr/c

In [2]:
sample_id_samui = "S1"

In [3]:
out_dir = here('samui_example')

In [5]:
sdata["table"]

AnnData object with n_obs × n_vars = 162254 × 377
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'region', 'z_level', 'nucleus_count'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [6]:
#   Read in AnnData and subset to this sample
adata = sdata["table"]
#path_groups = adata.obs['path_groups'].cat.categories
# adata = adata[adata.obs['sample_id'] == sample_id_spaceranger, :]
adata.obs.index.name = "barcode"


In [7]:
adata.var.index

Index(['ABCC11', 'ACE2', 'ACKR1', 'ACTA2', 'ACTG2', 'ADAM28', 'ADAMTS1',
       'ADGRE1', 'ADGRL4', 'ADH1C',
       ...
       'TRAC', 'TREM2', 'TSPAN19', 'UBE2C', 'UMOD', 'UPK3B', 'VCAN', 'VSIG4',
       'VWA5A', 'VWF'],
      dtype='object', length=377)

In [8]:

#   Convert the sparse gene-expression matrix to pandas DataFrame, with the
#   gene symbols as column names
gene_df = pd.DataFrame(
    adata.X.toarray(),
    index = adata.obs.index,
    columns = adata.var.index
)


In [9]:

#   Some gene symbols are actually duplicated. Just take the first column in
#   any duplicated cases
gene_df = gene_df.loc[: , ~gene_df.columns.duplicated()].copy()


In [10]:

#   Samui seems to break when using > ~ 5,000 genes. Take just the genes where
#   at least 10% of spots have nonzero counts
# gene_df = gene_df.loc[:, np.sum(gene_df > 0, axis = 0) > (gene_df.shape[0] * 0.1)].copy()

assert default_gene in gene_df.columns, "Default gene not in AnnData"

print('Using {} genes as features.'.format(gene_df.shape[1]))


Using 377 genes as features.


In [11]:

################################################################################
#   Split 'path_groups' column into binary columns for each of its values
################################################################################

#   Circumvent a Samui bug (https://github.com/chaichontat/samui/issues/84);
#   turn the categorical column 'path_groups' into several numeric columns with
#   just values of 0 and 1
# path_df = pd.DataFrame()
# for path_group in path_groups:
#     path_df[path_group] = (adata.obs['path_groups'] == path_group).astype(int)

################################################################################
#   Use the Samui API to create the importable directory for this sample
################################################################################

this_sample = Sample(name = sample_id_samui, path = out_dir)


In [18]:
sdata["table"].obs.index

Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       ...
       '162244', '162245', '162246', '162247', '162248', '162249', '162250',
       '162251', '162252', '162253'],
      dtype='object', name='barcode', length=162254)

In [14]:
pd.DataFrame(adata.obsm["spatial"], columns = ["x", "y"]).index

RangeIndex(start=0, stop=162254, step=1)

In [22]:

coords_df = pd.DataFrame(
    adata.obsm["spatial"], 
    index = adata.obs.index,
    columns = ["x", "y"]
    )

coords_df.index = coords_df.index.astype(str)

this_sample.add_coords(
    coords_df,
    name = "coords",  size = 2e-5 , mPerPx = 1e-6
)
 


Sample(name=S1, path=/Users/kevinstachelek/samui_example) with ['coords'] as coords and None as features.

In [23]:
#   Add the IF image for this sample
#this_sample.add_image(
#    tiff = img_path, channels = img_channels, scale = 1e-6,
#    defaultChannels = default_channels
#)

#   Add gene expression results (multiple columns) as a feature
this_sample.add_chunked_feature(
    gene_df, name = "Genes", coordName = "coords", dataType = "quantitative"
)


Sample(name=S1, path=/Users/kevinstachelek/samui_example) with ['coords'] as coords and ['Genes'] as features.

In [24]:
adata.obs.columns

Index(['cell_id', 'transcript_counts', 'control_probe_counts',
       'control_codeword_counts', 'unassigned_codeword_counts',
       'deprecated_codeword_counts', 'total_counts', 'cell_area',
       'nucleus_area', 'region', 'z_level', 'nucleus_count'],
      dtype='object')

In [25]:

#   Add additional requested observational columns (colData columns)
# this_sample.add_csv_feature(
#     adata.obs["Banksy_cluster"], name = "Banksy-Cluster", coordName = "coords",
#     dataType = "categorical"
# )

# this_sample.add_csv_feature(
#     adata.obs["banksy_cluster_group"], name = "Banksy Cluster Grouped", coordName = "coords",
#     dataType = "categorical"
# )

# this_sample.add_csv_feature(
#     adata.obs["domain_of_cell"], name = "Domain of Cell", coordName = "coords",
#     dataType = "categorical"
# )

this_sample.add_csv_feature(
    adata.obs["transcript_counts"], name = "nCounts", coordName = "coords",
    dataType = "quantitative"
)

# this_sample.add_csv_feature(
#     adata.obs["nGenes"], name = "nGenes", coordName = "coords",
#     dataType = "quantitative"
# )

this_sample.add_csv_feature(
    adata.obs["cell_area"], name = "Cell Area", coordName = "coords",
    dataType = "quantitative"
)

this_sample.add_csv_feature(
    adata.obs["nucleus_area"], name = "Nucleus Area", coordName = "coords",
    dataType = "quantitative"
)

#   Add pathology groups
# this_sample.add_csv_feature(
#     path_df, name = "Pathology Group", coordName = "coords",
#     dataType = "quantitative"
# )


Sample(name=S1, path=/Users/kevinstachelek/samui_example) with ['coords'] as coords and ['Genes', 'nCounts', 'Cell Area', 'Nucleus Area'] as features.

In [26]:

this_sample.set_default_feature(group = "Genes", feature = default_gene)



Sample(name=S1, path=/Users/kevinstachelek/samui_example) with ['coords'] as coords and ['Genes', 'nCounts', 'Cell Area', 'Nucleus Area'] as features.

In [27]:
this_sample.add_coords()

TypeError: Sample.add_coords() missing 1 required positional argument: 'df'

In [28]:
# df.index = df.index.astype(str)

this_sample?

Type:            Sample
String form:     name='S1' imgParams=None coordParams=[CoordParams(url=Url(url='coords.csv', type='local'), name=' <...>  ('Add csv feature Nucleus Area', <function Sample.add_csv_feature.<locals>.run at 0x3792cc670>)]
File:            ~/TOOLS/samui/loopy/sample.py
Docstring:       <no docstring>
Class docstring:
!!! abstract "Usage Documentation"
    [Models](../concepts/models.md)

A base class for creating Pydantic models.

Attributes:
    __class_vars__: The names of the class variables defined on the model.
    __private_attributes__: Metadata about the private attributes of the model.
    __signature__: The synthesized `__init__` [`Signature`][inspect.Signature] of the model.

    __pydantic_complete__: Whether model building is completed, or if there are still undefined fields.
    __pydantic_core_schema__: The core schema of the model.
    __pydantic_custom_init__: Whether the model has a custom `__init__` function.
    __pydantic_decorators__: Metadata 

In [ ]:

this_sample.write()


[08:59:27] INFO     'S1' Executing queued functions                                                    ]8;id=4740;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=295841;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

           INFO     S1 Adding coords 'coords'                                                          ]8;id=852862;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=268302;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

           INFO     S1 Adding chunked feature 'Genes'                                                  ]8;id=851221;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=446255;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

[09:00:15] INFO     S1 Concatenating and compressing chunks                                            ]8;id=384498;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=405991;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

[09:00:36] INFO     Writing compressed chunks for Genes: 5583529 bytes                                 ]8;id=586466;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=941913;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

           INFO     S1 Adding csv feature 'nCounts'                                                    ]8;id=351347;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=555501;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

           INFO     S1 Adding csv feature 'Cell Area'                                                  ]8;id=334114;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=182823;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

           INFO     S1 Adding csv feature 'Nucleus Area'                                               ]8;id=223402;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=755223;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

           INFO     'S1' written to /Users/kevinstachelek/samui_example                                ]8;id=566727;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py\logger.py]8;;\:]8;id=221810;file:///Users/kevinstachelek/TOOLS/samui/loopy/logger.py#27\27]8;;\

Sample(name=S1, path=/Users/kevinstachelek/samui_example) with ['coords'] as coords and ['Genes', 'nCounts', 'Cell Area', 'Nucleus Area'] as features.